# 🖍️ ShinChan Life Simulator - GRPO Training with OpenEnv
Train a small language model to make decisions as Shin-chan using Reinforcement Learning!

In [ ]:
# Bootstrap repo + dependencies (robust Colab setup)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Sarthaks-24/sinchan_env.git"
REPO_DIR = Path("/content/sinchan_env")

# 1) Core dependencies
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "openenv-core[core]>=0.2.2",
    "trl",
    "datasets",
    "torch",
    "transformers",
    "matplotlib",
])

# 2) Ensure repository exists in /content
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

# 3) Validate project root and editable install
if not (REPO_DIR / "pyproject.toml").exists():
    raise FileNotFoundError(f"pyproject.toml not found at {REPO_DIR}. Check REPO_URL.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
os.chdir(REPO_DIR)

print("Repository:", REPO_DIR)
print("Working dir:", os.getcwd())
print("Setup complete.")

### Step 1: Connect to the Environment
Replace the URL below with your Hugging Face Space URL.

Tip: use the `https://<username>-<space-name>.hf.space` runtime URL (not the repo URL).

In [ ]:
import os
# Set to your deployed Hugging Face Space URL
os.environ["ENV_URL"] = "https://gladiator-codes-sinchan-env.hf.space"
print("ENV_URL:", os.environ["ENV_URL"])

In [ ]:
# Quick health check against deployed Space
import requests, os

health_url = os.environ["ENV_URL"].rstrip("/") + "/health"
resp = requests.get(health_url, timeout=30)
print("Health URL:", health_url)
print("Status:", resp.status_code)
print("Body:", resp.text[:200])

### Step 2: Run Training (Configurable)

In [ ]:
# Run training with explicit args for reproducibility
import os

TRAIN_SCRIPT = "/content/sinchan_env/training/train_sinchan.py"
OUTPUT_DIR = "/content/sinchan_env/training/artifacts/run1"

if not os.path.exists(TRAIN_SCRIPT):
    raise FileNotFoundError(f"Training script not found: {TRAIN_SCRIPT}")

!python {TRAIN_SCRIPT} \
    --env-url "$ENV_URL" \
    --max-steps 200 \
    --dataset-size 200 \
    --learning-rate 1e-5 \
    --num-generations 2 \
    --output-dir {OUTPUT_DIR}

### Step 3: Generate Evaluation Summary
Run a quick baseline comparison and save JSON evidence.

In [ ]:
import os

EVAL_SCRIPT = "/content/sinchan_env/training/evaluate_scenarios.py"
EVAL_OUTPUT = "/content/sinchan_env/training/artifacts/eval_summary.json"

if not os.path.exists(EVAL_SCRIPT):
    raise FileNotFoundError(f"Eval script not found: {EVAL_SCRIPT}")

!python {EVAL_SCRIPT} \
    --env-url "$ENV_URL" \
    --episodes 10 \
    --output {EVAL_OUTPUT}

### Step 4: Generate Plot PNGs for Submission
Creates reward/loss/baseline comparison charts in `assets/`.

In [ ]:
import os

PLOT_SCRIPT = "/content/sinchan_env/training/plot_metrics.py"
RUN_DIR = "/content/sinchan_env/training/artifacts/run1"
EVAL_SUMMARY = "/content/sinchan_env/training/artifacts/eval_summary.json"
ASSETS_DIR = "/content/sinchan_env/assets"

if not os.path.exists(PLOT_SCRIPT):
    raise FileNotFoundError(f"Plot script not found: {PLOT_SCRIPT}")

!python {PLOT_SCRIPT} \
    --run-dir {RUN_DIR} \
    --eval-summary {EVAL_SUMMARY} \
    --assets-dir {ASSETS_DIR}